In [1]:
# Measure how quickly the system detects recovery (Crisis -> Elevated/Normal)

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")

# Load saved data
prices = pd.read_csv(f"{data_path}/prices.csv", index_col="Date", parse_dates=True)
regime = pd.read_csv(f"{data_path}/regime_states.csv", parse_dates=["date"]).set_index("date")

returns = prices.pct_change().dropna()

print("Loaded")
print(f"Prices: {prices.shape}")
print(f"Regime: {regime.shape}")
print(regime["regime_label"].value_counts())

Loaded
Prices: (2151, 5)
Regime: (2115, 2)
regime_label
Normal      1107
Crisis       586
Elevated     422
Name: count, dtype: int64


In [2]:
# Find all Crisis -> non-Crisis transition points
def find_recovery_transitions(regime_series):
    transitions = []
    regime_vals = regime_series["regime"].values
    dates = regime_series.index

    for i in range(1, len(regime_vals)):
        prev_regime = regime_vals[i - 1]
        curr_regime = regime_vals[i]

        # Crisis (2) -> Elevated (1) or Normal (0)
        if prev_regime == 2 and curr_regime != 2:
            transitions.append({
                "transition_date": dates[i],
                "from_regime": "Crisis",
                "to_regime": "Elevated" if curr_regime == 1 else "Normal"
            })

    return pd.DataFrame(transitions)

recovery_transitions = find_recovery_transitions(regime)

print(f"Found {len(recovery_transitions)} Crisis -> recovery transitions")
print(recovery_transitions)

Found 9 Crisis -> recovery transitions
  transition_date from_regime to_regime
0      2018-02-02      Crisis  Elevated
1      2019-01-29      Crisis  Elevated
2      2021-03-10      Crisis  Elevated
3      2021-12-17      Crisis  Elevated
4      2022-03-23      Crisis  Elevated
5      2022-08-02      Crisis  Elevated
6      2023-01-06      Crisis  Elevated
7      2025-05-09      Crisis  Elevated
8      2026-04-07      Crisis  Elevated


In [3]:
# For each recovery transition, find when AAPL price actually bottomed out and started recovering (rolling 5-day minimum as the "bottom" proxy)

def find_price_bottom_before(transition_date, prices, ticker="AAPL", lookback_days=30):
    """
    Look back from the transition date and find the actual local price bottom
    (lowest close in the lookback window).
    """
    window = prices[ticker].loc[
        transition_date - pd.Timedelta(days=lookback_days) : transition_date
    ]
    if window.empty:
        return None
    bottom_date = window.idxmin()
    return bottom_date

results = []
for _, row in recovery_transitions.iterrows():
    t_date = row["transition_date"]
    bottom_date = find_price_bottom_before(t_date, prices)

    if bottom_date is not None:
        # Positive lead_days = HMM detected recovery BEFORE price bottomed
        # Negative lead_days = HMM detected recovery AFTER price already bottomed (lagging)
        lead_days = (t_date - bottom_date).days

        results.append({
            "transition_date": t_date,
            "price_bottom_date": bottom_date,
            "lead_days": lead_days
        })

recovery_results = pd.DataFrame(results)
print(recovery_results)
print(f"\nAverage lead time: {recovery_results['lead_days'].mean():.1f} days")
print(f"(Positive = HMM detected recovery before price bottomed; Negative = HMM lagged the actual bottom)")

  transition_date price_bottom_date  lead_days
0      2018-02-02        2018-02-02          0
1      2019-01-29        2019-01-03         26
2      2021-03-10        2021-03-08          2
3      2021-12-17        2021-11-17         30
4      2022-03-23        2022-03-14          9
5      2022-08-02        2022-07-05         28
6      2023-01-06        2023-01-05          1
7      2025-05-09        2025-04-10         29
8      2026-04-07        2026-03-30          8

Average lead time: 14.8 days
(Positive = HMM detected recovery before price bottomed; Negative = HMM lagged the actual bottom)


In [4]:
# Retrain HMM with a shorter smoothing window and compare recovery detection speed
from hmmlearn import hmm
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import StandardScaler

macro_daily = pd.read_csv(f"{data_path}/macro_daily.csv", index_col="DATE", parse_dates=True)

def train_hmm_v3_custom_smooth(returns, macro_daily, ticker="AAPL", smooth_window=30):
    """Same as train_hmm_v3 in 04_hmm_regime.ipynb, but with a configurable smoothing window."""
    hmm_data = returns[[ticker]].copy()
    hmm_data["volatility"] = returns[ticker].rolling(20).std()
    hmm_data = hmm_data.join(macro_daily[["VIX", "Spread"]]).dropna()
    dates = hmm_data.index

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(hmm_data.values)
    vix = macro_daily["VIX"].reindex(dates)

    model = hmm.GaussianHMM(n_components=3, covariance_type="full",
                             n_iter=300, random_state=42, init_params="mc")
    model.startprob_ = np.array([0.6, 0.3, 0.1])
    model.transmat_ = np.array([
        [0.95, 0.04, 0.01],
        [0.05, 0.90, 0.05],
        [0.02, 0.08, 0.90],
    ])
    model.fit(X_scaled)
    states = model.predict(X_scaled)

    smoothed = uniform_filter1d(states.astype(float), size=smooth_window)
    smoothed = np.round(smoothed).astype(int)
    state_series = pd.Series(smoothed, index=dates)

    state_vix_means = {s: vix[state_series == s].mean() for s in range(3)}
    sorted_states = sorted(state_vix_means, key=state_vix_means.get)
    label_map = {sorted_states[0]: 0, sorted_states[1]: 1, sorted_states[2]: 2}
    state_series = state_series.map(label_map)

    return state_series

# Retrain with a shorter window (15 days instead of 30)
state_series_15 = train_hmm_v3_custom_smooth(returns, macro_daily, smooth_window=15)
regime_15 = pd.DataFrame({
    "regime": state_series_15.values
}, index=state_series_15.index)

print("Regime proportions (15-day smoothing):")
labels = {0: "Normal", 1: "Elevated", 2: "Crisis"}
for state, label in labels.items():
    ratio = (state_series_15 == state).mean() * 100
    print(f"  {label}: {ratio:.1f}%")

Regime proportions (15-day smoothing):
  Normal: 46.4%
  Elevated: 33.3%
  Crisis: 20.3%


In [5]:
# Repeat the recovery detection analysis with the 15-day smoothed regime
def find_recovery_transitions_v2(regime_series_values, dates):
    transitions = []
    for i in range(1, len(regime_series_values)):
        if regime_series_values[i - 1] == 2 and regime_series_values[i] != 2:
            transitions.append(dates[i])
    return transitions

transitions_15 = find_recovery_transitions_v2(state_series_15.values, state_series_15.index)

print(f"Found {len(transitions_15)} Crisis -> recovery transitions (15-day smoothing)")

results_15 = []
for t_date in transitions_15:
    bottom_date = find_price_bottom_before(t_date, prices)
    if bottom_date is not None:
        lead_days = (t_date - bottom_date).days
        results_15.append({
            "transition_date": t_date,
            "price_bottom_date": bottom_date,
            "lead_days": lead_days
        })

recovery_results_15 = pd.DataFrame(results_15)
print(recovery_results_15)
print(f"\nAverage lead time (15-day smoothing): {recovery_results_15['lead_days'].mean():.1f} days")
print(f"Average lead time (30-day smoothing): {recovery_results['lead_days'].mean():.1f} days")

Found 7 Crisis -> recovery transitions (15-day smoothing)
  transition_date price_bottom_date  lead_days
0      2019-01-28        2019-01-03         25
1      2020-11-06        2020-11-02          4
2      2022-03-21        2022-03-14          7
3      2022-07-28        2022-06-30         28
4      2022-12-09        2022-11-09         30
5      2025-05-07        2025-04-08         29
6      2026-04-01        2026-03-30          2

Average lead time (15-day smoothing): 17.9 days
Average lead time (30-day smoothing): 14.8 days


In [6]:
# Improved bottom detection: find the point after which a sustained recovery trend begins (not just the single lowest price)

def find_true_recovery_start(transition_date, prices, ticker="AAPL",
                               lookback_days=45, confirm_days=10):
    """
    Find the actual start of a sustained recovery:
    - Look back from the transition date
    - Find candidate low points
    - Confirm a low is a 'true bottom' if price rises for at least
      `confirm_days` days afterward without dropping below it again
    """
    window = prices[ticker].loc[
        transition_date - pd.Timedelta(days=lookback_days) :
        transition_date + pd.Timedelta(days=confirm_days)
    ]
    if window.empty or len(window) < confirm_days + 1:
        return None

    # Search candidate bottoms in chronological order
    for i in range(len(window) - confirm_days):
        candidate_date = window.index[i]
        candidate_price = window.iloc[i]

        # Only consider candidates before or at the transition date
        if candidate_date > transition_date:
            break

        future_prices = window.iloc[i+1 : i+1+confirm_days]

        # Confirm: all following prices stay at or above the candidate (sustained recovery)
        if (future_prices >= candidate_price * 0.98).all():  # allow 2% noise tolerance
            return candidate_date

    return None

# Re-run recovery detection with the improved bottom-finding logic
results_v3 = []
for _, row in recovery_transitions.iterrows():
    t_date = row["transition_date"]
    true_bottom = find_true_recovery_start(t_date, prices)

    if true_bottom is not None:
        lead_days = (t_date - true_bottom).days
        results_v3.append({
            "transition_date": t_date,
            "true_bottom_date": true_bottom,
            "lead_days": lead_days
        })

recovery_results_v3 = pd.DataFrame(results_v3)
print(recovery_results_v3)
print(f"\nAverage lead time (improved method, 30-day smoothing): {recovery_results_v3['lead_days'].mean():.1f} days")
print(f"Original naive method (30-day smoothing): {recovery_results['lead_days'].mean():.1f} days")

  transition_date true_bottom_date  lead_days
0      2018-02-02       2018-01-02         31
1      2019-01-29       2019-01-03         26
2      2021-03-10       2021-01-29         40
3      2021-12-17       2021-11-02         45
4      2022-03-23       2022-02-23         28
5      2022-08-02       2022-06-21         42
6      2023-01-06       2022-11-29         38
7      2025-05-09       2025-04-08         31
8      2026-04-07       2026-03-13         25

Average lead time (improved method, 30-day smoothing): 34.0 days
Original naive method (30-day smoothing): 14.8 days


In [7]:
print("""
Summary of findings:

Unlike crisis detection (measured against a clear threshold crossing),
recovery detection lead time is highly sensitive to how "the bottom" is
defined:

  - Naive single-day minimum:           14.8 days
  - Shorter (15-day) regime smoothing:  17.9 days
  - Sustained-recovery confirmation:    34.0 days

This instability suggests that "recovery" is a fundamentally fuzzier
concept than "crisis onset" — crisis has a clear quantitative trigger
(risk score crossing 0.5), but recovery requires defining what counts
as a sustained trend reversal, which is inherently more subjective.

A more robust future approach might define recovery using the same
type of clear threshold used for crisis detection (e.g. risk score
falling and staying below 0.3 for N consecutive days) rather than
inferring it from price action alone.
""")


Summary of findings:

Unlike crisis detection (measured against a clear threshold crossing),
recovery detection lead time is highly sensitive to how "the bottom" is
defined:

  - Naive single-day minimum:           14.8 days
  - Shorter (15-day) regime smoothing:  17.9 days
  - Sustained-recovery confirmation:    34.0 days

This instability suggests that "recovery" is a fundamentally fuzzier
concept than "crisis onset" — crisis has a clear quantitative trigger
(risk score crossing 0.5), but recovery requires defining what counts
as a sustained trend reversal, which is inherently more subjective.

A more robust future approach might define recovery using the same
type of clear threshold used for crisis detection (e.g. risk score
falling and staying below 0.3 for N consecutive days) rather than
inferring it from price action alone.

